#### Pretrain

##### Настройка окружения

In [14]:
import os
import glob
import re
import nltk
import random
import pandas as pd
import numpy as np
from tqdm import tqdm
import accelerate
import jinja2
import hashlib
from difflib import SequenceMatcher

import torch
from sklearn.model_selection import train_test_split
from transformers import (
    PreTrainedTokenizerFast,
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    LlamaConfig,
    LlamaForCausalLM,
    Trainer,
    TrainingArguments,
    TrainerCallback,
    DataCollatorForLanguageModeling,
    EarlyStoppingCallback)

from datasets import Dataset, load_dataset

from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace

from trl import SFTTrainer, SFTConfig
from peft import LoraConfig, get_peft_model


In [3]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"using device: {device}")

using device: cuda


In [56]:
if not os.path.exists("RussianNovels"):
    !git clone https://github.com/JoannaBy/RussianNovels.git

file_paths = glob.glob('RussianNovels/corpus/*.txt')
print(f"Найдено файлов: {len(file_paths)}")

Найдено файлов: 108


In [57]:
# чтение файлов в 1 список
raw_texts = []
for file_path in file_paths:
    with open(file_path, 'r', encoding='utf-8') as f:
        try:
            text = f.read()
            raw_texts.append(text)
        except Exception as e:
            print(f"Ошибка чтения файла {file_path}: {e}")
            
print(f"Загружено текстов: {len(raw_texts)}")

Загружено текстов: 108


##### 1. Препроцессинг данных

In [58]:
nltk.download('punkt')
nltk.download('punkt_tab')

# фиксим кириллицу
def is_cyrillic(text):
    return not bool(re.search('[a-zA-Z]', text))

# фиксим пунктуацию
def clean_text(text):
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'\s+([.,!?;:])', r'\1', text)
    text = re.sub(r'[,]{2,}', ',', text)
    text = re.sub(r'[!]{2,}', '!', text)
    text = re.sub(r'[?]{2,}', '?', text)
    text = re.sub(r'--+', '—', text)
    text = text.replace("—", " — ")
    text = re.sub(r'\s+', ' ', text)
    text = text.strip()
    return text

all_sentences = []

for text in tqdm(raw_texts):
    sentences = nltk.sent_tokenize(text, language='russian')
    
    for sent in sentences:
        sent = clean_text(sent)
        if len(sent) > 10 and is_cyrillic(sent):
            all_sentences.append(sent)
            
# фиксим дубликаты
orig_count = len(all_sentences)
uniq_sentences = list(set(all_sentences))
print(f"Предложений до очистки: {orig_count}")
print(f"Предложений после удаления дубликатов: {len(uniq_sentences)}")

[nltk_data] Downloading package punkt to /home/ubuntu/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /home/ubuntu/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
100%|██████████| 108/108 [00:17<00:00,  6.22it/s]

Предложений до очистки: 470932
Предложений после удаления дубликатов: 454666


In [59]:
# сборка в чанки
full_clean_text = " ".join(uniq_sentences)
chunk_size = 1400

chunks = [full_clean_text[i:i+chunk_size] for i in range(0, len(full_clean_text), chunk_size)]

print(f"Сформировано чанков: {len(chunks)}")

Сформировано чанков: 29369


##### 2. Обучение токенизатора

In [60]:
tokenizer = Tokenizer(BPE(unk_token="<unk>"))
tokenizer.pre_tokenizer = Whitespace()

trainer = BpeTrainer(
    vocab_size=3000, 
    special_tokens=["<bos>", "<eos>", "<unk>", "<pad>"],
    min_frequency=2
)

tokenizer.train_from_iterator(chunks, trainer)

tokenizer.save("custom_tokenizer.json")

fast_tokenizer = PreTrainedTokenizerFast(
    tokenizer_file="custom_tokenizer.json",
    bos_token="<bos>",
    eos_token="<eos>",
    unk_token="<unk>",
    pad_token="<pad>"
)

sample_text = "Велик был год и страшен год по рождестве Христовом 1918"
tokens = fast_tokenizer.encode(sample_text)
print(f"Пример токенизации: {fast_tokenizer.convert_ids_to_tokens(tokens)}")
print(f"Размер словаря: {fast_tokenizer.vocab_size}")




Пример токенизации: ['Ве', 'ли', 'к', 'был', 'год', 'и', 'стра', 'шен', 'год', 'по', 'ро', 'жде', 'стве', 'Хри', 'сто', 'вом', '1', '9', '1', '8']
Размер словаря: 3000


Собственный токенизатор (vocab=3000) гораздо эффективнее для этой задачи. Каждый токен будет встречаться достаточно часто, чтобы модель выучила его эмбеддинг. Использование BPE позволяет эффективно кодировать редкие слова через подслова.


##### 3. Подготовка данных к претрейну

In [61]:
context_length = 512

def tokenization_func(examples):
    # добавление bos, eos к тексту
    text_list = [fast_tokenizer.bos_token + t + fast_tokenizer.eos_token for t in examples['text']]
    
    # с обрезкой и паддингом
    return fast_tokenizer(
        text_list,
        truncation=True,
        max_length=context_length,
        padding="max_length"
    )
    
dataset = Dataset.from_dict({"text": chunks})

tokenized_dataset = dataset.map(
    tokenization_func,
    batched=True,
    num_proc=4,
    remove_columns=["text"]
)

# разбиваем на подвыборки
split_dataset = tokenized_dataset.train_test_split(test_size=.05)
train_dataset = split_dataset['train']
eval_dataset = split_dataset['test']

print(train_dataset)

Map (num_proc=4): 100%|██████████| 29369/29369 [00:08<00:00, 3373.84 examples/s]

Dataset({
    features: ['input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 27900
})


##### 4. Инициализация модели

In [62]:
config = LlamaConfig(
    vocab_size=fast_tokenizer.vocab_size,
    hidden_size=1024,
    intermediate_size=1536,
    num_hidden_layers=16,
    num_attention_heads=16,
    num_key_value_heads=8,
    max_position_embeddings=context_length,
    bos_token_id=fast_tokenizer.bos_token_id,
    eos_token_id=fast_tokenizer.eos_token_id,
    pad_token_id=fast_tokenizer.pad_token_id
)

model = LlamaForCausalLM(config)

# смотрим параметры
n_params = sum(p.numel() for p in model.parameters())
print(f"кол-во параметров модели: {n_params / 1e6:.2f}M")

кол-во параметров модели: 132.01M


##### 5. Обучение и валидация качества

In [63]:
test_prompts = [
    "Все мысли, которые имеют огромные последствия",
    "Сила войска зависит от его духа",
    "Мысль о том, что он принес страдания",
    "Человек сознает себя свободным",
    "Что бы ни случилось, я всегда буду",
    "Любовь мешает смерти",
    "Нет, жизнь не кончена",
    "Всякая мысль, даже самая простая",
    "Война не любезность, а самое гадкое дело",
    "Чтобы жить честно"
]

In [64]:
class GenerationCallback(TrainerCallback):
    # колбэк для генерации текста в процессе обучения чтобы видеть прогресс модели
    def __init__(self, tokenizer, model, prompts):
        self.tokenizer = tokenizer
        self.model = model
        self.prompts = prompts
        
    def on_evaluate(self, args, state, control, **kwargs):
        print("\n=== GENERATION LOG (Validation Phase) ===")
        device = next(self.model.parameters()).device
        self.model.eval()
        
        for prompt in self.prompts[:2]: 
            inputs = self.tokenizer(prompt, return_tensors="pt", return_token_type_ids=False).to(device)
            with torch.no_grad():
                outputs = self.model.generate(
                    **inputs, 
                    max_new_tokens=40, 
                    do_sample=True, 
                    temperature=0.6,
                    top_k = 50,
                    top_p = .95,
                    repetition_penalty=1.2,
                    no_repeat_ngram_size=2,
                    pad_token_id=self.tokenizer.pad_token_id,
                    eos_token_id=self.tokenizer.eos_token_id
                )
            raw_output = self.tokenizer.decode(outputs[0], skip_special_tokens=True)
            readable_output = raw_output.replace(" ", "")
            readable_output = readable_output.replace(",", ", ").replace(".", ". ").replace("!", "! ").replace("?", "? ").replace("—", " — ")
            generated_text = self.tokenizer.decode(outputs[0], skip_special_tokens=True)
            print(f"Input: {prompt}\nRaw_Output: {raw_output}\nReadable_output: {readable_output}\n")
        
        self.model.train()

training_args = TrainingArguments(
    output_dir="./russian_lit_model",
    overwrite_output_dir=True,
    num_train_epochs=15,
    per_device_train_batch_size=64,  
    per_device_eval_batch_size=64,    
    eval_strategy="steps", 
    eval_steps=250,
    save_steps=250,
    save_total_limit=1,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    load_best_model_at_end=True,
    logging_steps=100,
    learning_rate=1e-4,
    weight_decay=0.01,
    warmup_ratio=.1,              
    fp16=torch.cuda.is_available(), 
    report_to="none"
)

data_collator = DataCollatorForLanguageModeling(
    tokenizer=fast_tokenizer, 
    mlm=False
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
    callbacks=[
        GenerationCallback(fast_tokenizer, model, test_prompts),
        EarlyStoppingCallback(early_stopping_patience=3)
    ]
    
)

trainer.train()
trainer.save_model("./russian_lit_model_final")
fast_tokenizer.save_pretrained("./russian_lit_model_final")

Step,Training Loss,Validation Loss
250,6.492100,6.033516
500,5.040300,4.872121
750,4.481500,4.295129
1000,4.024800,4.050580
1250,3.920600,3.913920
1500,3.717900,3.839208
1750,3.684800,3.781510
2000,3.523700,3.757555
2250,3.473100,3.745987
2500,3.339400,3.728899



=== GENERATION LOG (Validation Phase) ===
Input: Все мысли, которые имеют огромные последствия
Raw_Output: Все мысли , которые име ют огром ные послед ствия по ший — сказал
Readable_output: Всемысли, которыеимеютогромныепоследствияпоший — сказал

Input: Сила войска зависит от его духа
Raw_Output: Си ла вой ска зави си т от его ду ха гах по чер з пы и , а с су х от ки : " – За х о сла пе ну . Я вы не от меня , что это при че в то , если я не су
Readable_output: Силавойсказависитотегодухагахпочерзпыи, ассухотки:"–Захослапену. Явынеотменя, чтоэтопричевто, еслиянесу


=== GENERATION LOG (Validation Phase) ===
Input: Все мысли, которые имеют огромные последствия
Raw_Output: Все мысли , которые име ют огром ные послед ствия и не вы ходит , как я на ле во му . — Она вас не при ш лось , что ли ? – Я не было бы и ничего не будет . - Это не знал , - сказал
Readable_output: Всемысли, которыеимеютогромныепоследствияиневыходит, какяналевому.  — Онаваснепришлось, чтоли? –Янебылобыиничегонебудет. -Э

('./russian_lit_model_final/tokenizer_config.json',
 './russian_lit_model_final/special_tokens_map.json',
 './russian_lit_model_final/tokenizer.json')

попыталась вывести читабельный вывод генерации на валидации через readable_output. лучше это конечно убрать, т.к. только мешает нормально посмотреть ход обучения.


##### 6. Финальная генерация ответов на запросы

In [69]:
print("\n=== ИТОГОВАЯ ГЕНЕРАЦИЯ ОТВЕТОВ ===")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

for prompt in test_prompts:
    inputs = fast_tokenizer(prompt, return_tensors="pt", return_token_type_ids=False).to(device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs, 
            max_new_tokens=50, 
            do_sample=True, 
            temperature=0.2,       
            top_k=40,
            top_p=0.5,
            repetition_penalty=1.2, 
            no_repeat_ngram_size=2,
            pad_token_id=fast_tokenizer.pad_token_id,
            eos_token_id=fast_tokenizer.eos_token_id
        )
    text = fast_tokenizer.decode(outputs[0], skip_special_tokens=True)
    # clean_text_view = text.replace(" ", "").replace(",", ", ").replace(".", ". ").replace("!", "! ").replace("?", "? ").replace("—", " — ")
    
    print(f"PROMPT: {prompt}")
    print(f"OUTPUT: {text}")
    print("-" * 60)


=== ИТОГОВАЯ ГЕНЕРАЦИЯ ОТВЕТОВ ===
PROMPT: Все мысли, которые имеют огромные последствия
OUTPUT: Все мысли , которые име ют огром ные послед ствия ни в чем не по сти жи мо . — Я не знаю , что ты говори шь ! - А я так думаю : " Не ужели вы дума ете , как он сам винова т ?" Он ъ был ъ въ нем ъ и на г рѣ лъ
------------------------------------------------------------
PROMPT: Сила войска зависит от его духа
OUTPUT: Си ла вой ска зави си т от его ду ха стало стало про стала не пом ма ко про м в ше л сто ! — Я по зи , как вы дума ешь , я вас люблю , и все это вздо р . - А ты чего знаешь ? В то время как он под ходил к двери
------------------------------------------------------------
PROMPT: Мысль о том, что он принес страдания
OUTPUT: Мы с ль о том , что он при нес стра дания и по просил его . — Я не знаю , как ты дума ешь , — сказал он , у казы вая на пле чи . - А вот я тебе скажу : " Не ужели вы не вери те в бе д ную минуту ?" Он был очень до
---------------------------------------------

У модели получился относительно неплохой лосс к концу обучения - 2.94290 на тренировочном и 3.832294 на валидационном наборах. Была применена ранняя остановка обучения, чтобы модель не переобучалась. 
При генерации мы получаем нормальный и ожидаемый результат для pretrain. Ввиду маленького размера словаря BPE, модель вынуждена оперировать слогами, что может приводить к появлению "артефактов". Когда модель не уверена, она начинает перебирать наиболее вероятные слоги. Однако, модель хорошо выучила пунктуацию, стиль обращений, что означает, что модель научилась структуре языка. Это главное требование, которого мы добивались в pretrain-стадии обучения модели

#### SFT

##### 1. Подготовка датасета

In [18]:
raw_dataset = load_dataset("d0rj/alpaca-cleaned-ru")
raw_dataset

DatasetDict({
    train: Dataset({
        features: ['input', 'instruction', 'output'],
        num_rows: 51760
    })
})

In [24]:
# приведение к диалоговому формату
SYSTEM_PROMPT = "Ты — полезный, вежливый и русскоязычный ассистент."

def to_dialog(example):
    user_text = example["instruction"]
    if example.get("input"):
        user_text = user_text + "\n" + example["input"]

    return {
        "system": SYSTEM_PROMPT,
        "user": user_text.strip(),
        "assistant": example["output"].strip()
    }

dialog_dataset = raw_dataset["train"].map(
    to_dialog,
    remove_columns=raw_dataset["train"].column_names
)


Map: 100%|██████████| 51760/51760 [00:02<00:00, 19236.04 examples/s]


In [25]:
# деление на подвыборки
dataset = dialog_dataset.train_test_split(test_size=0.1, seed=42)
tmp = dataset["test"].train_test_split(test_size=0.5, seed=42)

train_ds = dataset["train"]
val_ds   = tmp["train"]
test_ds  = tmp["test"]

len(train_ds), len(val_ds), len(test_ds)


(46584, 2588, 2588)

In [26]:
# дедупликация данных
def example_hash(ex):
    text = ex["user"] + ex["assistant"]
    return hashlib.md5(text.encode("utf-8")).hexdigest()

def remove_overlap(train, *others):
    other_hashes = set()
    for ds in others:
        for ex in ds:
            other_hashes.add(example_hash(ex))

    filtered = [
        ex for ex in train
        if example_hash(ex) not in other_hashes
    ]
    return Dataset.from_list(filtered)

In [27]:
train_ds = remove_overlap(train_ds, val_ds, test_ds)

In [28]:
# внутренняя дедупликация
def fast_deduplicate(ds, max_len=200):
    seen = set()
    unique = []

    for ex in ds:
        key = (
            ex["user"][:max_len],
            ex["assistant"][:max_len]
        )
        if key not in seen:
            seen.add(key)
            unique.append(ex)

    return Dataset.from_list(unique)


In [29]:
train_ds = fast_deduplicate(train_ds)
val_ds   = fast_deduplicate(val_ds)
test_ds  = fast_deduplicate(test_ds)

len(train_ds), len(val_ds), len(test_ds)

(46574, 2588, 2588)

##### 2. Дообучение модели

In [30]:
model_name = "Qwen/Qwen2.5-0.5B"

tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    trust_remote_code=True
)
tokenizer.pad_token = tokenizer.eos_token

In [32]:
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)

In [33]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj", "k_proj", "v_proj",
        "o_proj", "gate_proj", "up_proj", "down_proj"
    ]
)

In [34]:
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 4,399,104 || all params: 498,431,872 || trainable%: 0.8826


In [35]:
# форматирование диалога для SFTTrainer
def format_dialog(example):
    return (
        f"<|system|>\n{example['system']}\n"
        f"<|user|>\n{example['user']}\n"
        f"<|assistant|>\n{example['assistant']}"
    )


In [37]:
# параметры обучения
training_args = TrainingArguments(
    output_dir="./qwen2.5-ru-sft-lora",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-5,
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    fp16=True,
    logging_steps=50,
    eval_strategy="steps",
    eval_steps=500,
    save_steps=500,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="none"
)

In [43]:
trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    formatting_func=format_dialog,
    # max_length=1024,
    args=training_args,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)


/home/ubuntu/.local/lib/python3.10/site-packages/transformers/training_args.py:2111: FutureWarning: `--push_to_hub_token` is deprecated and will be removed in version 5 of 🤗 Transformers. Use `--hub_token` instead.
  warnings.warn(
Applying formatting function to train dataset:   0%|          | 0/46574 [00:00<?, ? examples/s]

Truncating eval dataset: 100%|██████████| 2588/2588 [00:00<00:00, 435868.08 examples/s]
The model is already on multiple devices. Skipping the move to device specified in `args`.


In [ ]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss,Validation Loss
500,1.396800,1.338691


##### 3. Оценка адекватности генерации

In [ ]:
questions_rus = [
    "сколько планет в нашей солнечной системе?",
    "расскажи стих",
    "когда собирать крыжовник?",
    "Как быстро выучить новый язык?"
  ]

In [ ]:
def generate_answer(question):
    prompt = (
        "<|system|>\nТы — полезный русскоязычный ассистент.\n"
        "<|user|>\n" + question + "\n"
        "<|assistant|>\n"
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=200,
            do_sample=True,
            top_p=0.9,
            temperature=0.7
        )

    return tokenizer.decode(output[0], skip_special_tokens=True)


In [ ]:
for q in questions_rus:
    print("=" * 80)
    print("QUESTION:", q)
    print("ANSWER:")
    print(generate_answer(q))
